# RetrieveChat based FinRobot-RAG

In this demo, we showcase the RAG usecase of our finrobot, which inherits from autogen's RetrieveChat implementation.


Instead of using `RetrieveUserProxyAgent` directly, we register the context retrieval as a function for our bots.
For detailed implementation, refer to [rag function](../finrobot/functional/rag.py) and [rag workflow](../finrobot/agents/workflow.py) of `SingleAssistantRAG` 

In [1]:
import autogen
from finrobot.agents.workflow import SingleAssistantRAG

for openai configuration, rename OAI_CONFIG_LIST_sample to OAI_CONFIG_LIST and replace the api keys

In [2]:
# Read OpenAI API keys from a JSON file
llm_config = {
    "config_list": autogen.config_list_from_json(
        "../OAI_CONFIG_LIST",
        filter_dict={"model": ["qwen-plus"]},
    ),
    "timeout": 120,
    "temperature": 0,
}

From `finrobot.agents.workflow` we import the `SingleAssistantRAG`, which takes a `retrieve_config` as input.
For `docs_path`, we first put our generated pdf report from [this notebook](./agent_annual_report.ipynb). 

For more configuration, refer to [autogen's documentation](https://microsoft.github.io/autogen/docs/reference/agentchat/contrib/retrieve_user_proxy_agent)

Then, lets do a simple Q&A.

In [ ]:
assitant = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "vector_db": None,  # Autogen has bug for this version
        "docs_path": [
            "../report/Microsoft_Annual_Report_2023.pdf",
        ],
        "chunk_token_size": 1000,
        "get_or_create": True,
        "collection_name": "msft_analysis",
        "must_break_at_empty_line": False,
    },
)
assitant.chat("How's msft's 2023 income? Provide with some analysis.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


User_Proxy (to Data_Analyst):

How's msft's 2023 income? Provide with some analysis.

--------------------------------------------------------------------------------
Data_Analyst (to User_Proxy):


***** Suggested tool call (call_69f3f702ca9c407688b3e6): retrieve_content *****
Arguments: 
{"message": "Microsoft 2023 income statement and revenue analysis"}
*******************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION retrieve_content...
Call ID: call_69f3f702ca9c407688b3e6
Input arguments: {'message': 'Microsoft 2023 income statement and revenue analysis'}
Trying to create collection.


d:\pythonProject\FinRobot-master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


doc_ids:  [['doc_0']]
Adding content of doc doc_0 to context.
User_Proxy (to Data_Analyst):

***** Response from calling tool (call_69f3f702ca9c407688b3e6) *****
Below is the context retrieved from the required file based on your query.
If you can't answer the question with or without the current context, you should try using a more refined search query according to your requirements, or ask for more contexts.

Your current query is: Microsoft 2023 income statement and revenue analysis

Retrieved context is: Equity Research Report: Microsoft Corporation
FinRobot
https://ai4finance.org/
Income Summarization The company experienced a 7% Year-over-Year increase in revenue, driven by significant contributions from its Intelligent Cloud and Productivity and Business Processes segments, indicating a strong demand for cloud-based solutions and productivity software. Despite the revenue growth, the Cost of Goods Sold (COGS) increased by 5%, suggesting a need for closer cost control measures to

Here we come up with a more complex case, where we put the 10-k report of MSFT here.

Let' see how the agent work this out.

In [4]:
assitant = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "vector_db": None,  # Autogen has bug for this version
        "docs_path": [
            "../report/2023-07-27_10-K_msft-20230630.htm.pdf",
        ],
        "chunk_token_size": 2000,
        "collection_name": "msft_10k",
        "get_or_create": True,
        "must_break_at_empty_line": False,
    },
    rag_description="Retrieve content from MSFT's 2023 10-K report for detailed question answering.",
)
assitant.chat("How's msft's 2023 income? Provide with some analysis.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


User_Proxy (to Data_Analyst):

How's msft's 2023 income? Provide with some analysis.

--------------------------------------------------------------------------------
Data_Analyst (to User_Proxy):


***** Suggested tool call (call_96171e7c3be546bda34a66): retrieve_content *****
Arguments: 
{"message": "MSFT 2023 income statement analysis"}
*******************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION retrieve_content...
Call ID: call_96171e7c3be546bda34a66
Input arguments: {'message': 'MSFT 2023 income statement analysis'}


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Trying to create collection.


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


doc_ids:  [['doc_34', 'doc_1', 'doc_38']]
Adding content of doc doc_34 to context.
Adding content of doc doc_1 to context.
Adding content of doc doc_38 to context.
User_Proxy (to Data_Analyst):

***** Response from calling tool (call_96171e7c3be546bda34a66) *****
Below is the context retrieved from the required file based on your query.
If you can't answer the question with or without the current context, you should try using a more refined search query according to your requirements, or ask for more contexts.

Your current query is: MSFT 2023 income statement analysis

Retrieved context is: November 18, 2021 December 9, 2021 $ March 10, 2022 June 9, 2022 August 18, 2022 September 8, 2022
0.62 $ 0.62 0.62 0.62
4,652 4,645 4,632 4,621
December 7, 2021 March 14, 2022 June 14, 2022
February 17, 2022 May 19, 2022
Total
$
2.48 $
18,550
The dividend declared on June 13, 2023 was included in other current liabilities as of June 30, 2023.
90
PART II Item 8
NOTE 17 — ACCUMULATED OTHER COMPREHEN